# 0d · Self-referential (stance) vs descriptive sentiment — decomposing the reflection

**Motivation (from the discussion).** The whole-reflection sentiment blends two kinds of sentence: **descriptive** ('Randall seems devastated' — the *character's* state) and **stance** ('I feel bad for him' — the *viewer's* feeling). Whole-text sentiment tracks the character's `positive emotion` and misses `like` (~0.07). **Hypothesis:** that null is a *dilution* — split the reflection, score each subset, and stance-sentiment should recover `like` while descriptive-sentiment tracks `positive emotion`. If so, verbalized feeling IS recoverable; it's just concentrated in the ~20% stance sentences and drowned out by the descriptive majority.

Data: ~3,700 sentences, 53% descriptive / 47% first-person. **Scaffold — run 0d.3 locally (transformer).**

## 0d.1 · Load + sentence-split

In [3]:
# 0d.1 · Load reflections and split into sentences.
import pandas as pd, numpy as np, re
sc = pd.read_csv("results/scored/00__reflection_sentiment.csv")[["Participant","Run","Character","Raw_Text"]].copy()
sc = sc.dropna(subset=["Raw_Text"]); sc["Character"] = sc["Character"].str.lower().str.strip()
def split_sentences(t):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n+", str(t)) if s.strip()]
sent = sc.assign(sentence=sc["Raw_Text"].map(split_sentences)).explode("sentence").reset_index(drop=True)
sent = sent[sent["sentence"].str.split().str.len() >= 2]
print(f"{len(sc)} reflections -> {len(sent)} sentences ({len(sent)/len(sc):.1f} per reflection)")


1272 reflections -> 3606 sentences (2.8 per reflection)


## 0d.2 · Classify by EXPERIENCER (whose emotion is it?)

Your distinction — *"I'm confused why he would do that"* (speaker experiences it → **stance**) vs *"he was confused"* (character experiences it → **descriptive**) — is about **who holds the mental/emotional state**, i.e. the grammatical **experiencer**. That's a dependency-parse question, not subjectivity/stance, so we find the **subject of the affect word** with spaCy. Setup once: `pip install spacy && python -m spacy download en_core_web_sm`.

In [4]:
# 0d.2 · Classify by the EXPERIENCER of the mental/emotional state (your distinction):
#   "stance"      = the SPEAKER is the experiencer   -> "I'm confused why he would do that"
#   "descriptive" = the CHARACTER is the experiencer -> "he was confused that he did that"  (or no affect word)
# This is a SEMANTIC-ROLE question (experiencer = 1st vs 3rd person), so we DEPENDENCY-PARSE (spaCy) to find
# the grammatical SUBJECT of each affect/cognition predicate -- NOT a subjectivity/stance classifier.
# Setup (once):  pip install spacy  &&  python -m spacy download en_core_web_sm
import spacy
import spacy.cli

# Force the download using the active Jupyter Python environment
spacy.cli.download("en_core_web_sm")
nlp = spacy.load("en_core_web_sm")

FIRST = {"i", "we", "me", "us", "myself", "ourselves"}
# affect + cognition predicate lemmas (extend freely -- this lexicon defines "a mental/emotional state")
AFFECT = {"confuse","confused","feel","felt","sad","happy","angry","mad","upset","scared","afraid","fear",
          "worry","worried","anxious","nervous","frustrate","frustrated","surprise","surprised","hurt","glad",
          "proud","ashamed","guilty","embarrassed","excited","overwhelmed","emotional","stressed","depressed",
          "empathize","sympathize","relate","like","love","hate","care","pity","understand","think","believe",
          "hope","wish","enjoy","disappoint","disappointed","annoyed","curious","confident","calm","comfortable"}

def _subject_of(tok):
    # adjective/participle complements ("I am confused") hang their subject on the copula/head verb
    anchor = tok.head if tok.dep_ in ("acomp","attr","oprd","xcomp","ccomp") else tok
    for a in (anchor, anchor.head):
        subs = [c for c in a.children if c.dep_ in ("nsubj","nsubjpass")]
        if subs: return subs[0]
    return None

def experiencer_label(sentence):
    doc = nlp(str(sentence))
    for tok in doc:                                   # first affect predicate = the main mental state
        if tok.lemma_.lower() in AFFECT:
            subj = _subject_of(tok)
            if subj is None: continue
            w = subj.lemma_.lower()
            if w in FIRST or subj.text.lower() in FIRST:
                return "stance"                       # speaker is the experiencer
            return "descriptive"                      # a non-first-person subject -> the character
    return "descriptive"                              # no mental-state predicate -> pure description

sent = sent.reset_index(drop=True)
sent["kind"] = sent["sentence"].map(experiencer_label)
sent["kind_conf"] = float("nan")                       # (parse-based; confidence via the LLM/zero-shot variant)
import os; os.makedirs("results/scored", exist_ok=True)
sent[["kind"]].to_csv("results/scored/0d__sentence_labels.csv", index=False)
print(sent["kind"].value_counts())
print("\nExperiencer rule: subject of the affect/cognition word is 1st-person -> stance; else -> descriptive.")
print("Validate on a hand-labeled gold set; extend AFFECT for predicates it misses.")
# review:  sent[["kind","sentence"]].sample(30, random_state=0).to_string()


Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 54.8 MB/s  0:00:00eta 0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
kind
descriptive    2817
stance          789
Name: count, dtype: int64

Experiencer rule: subject of the affect/cognition word is 1st-person -> stance; else -> descriptive.
Validate on a hand-labeled gold set; extend AFFECT for predicates it misses.


## 0d.3 · Score each subset (RUN LOCALLY)

In [5]:
# 0d.3 · Score STANCE vs DESCRIPTIVE text separately, per (participant, run, character). RUN LOCALLY (transformer).
# Rejoin each subset's sentences into one text per cell, then score with the Step-0 winner (Twitter-RoBERTa).
import numpy as np, pandas as pd, os
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from scipy.special import softmax
_TW = "cardiffnlp/twitter-roberta-base-sentiment-latest"
_tok = AutoTokenizer.from_pretrained(_TW); _mdl = AutoModelForSequenceClassification.from_pretrained(_TW).eval()
def _val(text):
    if not str(text).strip(): return np.nan
    p = softmax(_mdl(**_tok(str(text), return_tensors="pt", truncation=True, max_length=512)).logits.detach().numpy()[0])
    return float(p[2] - p[0])                      # pos - neg
rows = []
for (P, R, C), g in sent.groupby(["Participant","Run","Character"]):
    d = {"Participant": P, "Run": R, "Character": C}
    for kind in ["stance", "descriptive"]:
        joined = " ".join(g[g.kind == kind]["sentence"].tolist())
        d[f"val_{kind}"] = _val(joined); d[f"n_{kind}"] = int((g.kind == kind).sum())
    rows.append(d)
split = pd.DataFrame(rows)
os.makedirs("results/scored", exist_ok=True)
split.to_csv("results/scored/0d__split_sentiment.csv", index=False)
print("wrote results/scored/0d__split_sentiment.csv", split.shape)
print("cells with >=1 stance sentence:", int((split.n_stance>0).sum()), "/", len(split))


/Users/rheamadhogarhia/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


wrote results/scored/0d__split_sentiment.csv (1271, 7)
cells with >=1 stance sentence: 577 / 1271


## 0d.4 · Validation — the decisive test

In [6]:
# 0d.4 · VALIDATION — the key test. Does STANCE-sentiment track LIKE, and DESCRIPTIVE track POSITIVE EMOTION?
# Group-level (same grid as Step 1). If stance-sentiment recovers 'like' where whole-text sentiment could not,
# that CONFIRMS the whole-text null on liking was a dilution by descriptive sentences -- verbalized feeling IS
# recoverable, just concentrated in the minority stance sentences.
import pandas as pd, numpy as np, re
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, LeaveOneGroupOut
from scipy.stats import spearmanr
split = pd.read_csv("results/scored/0d__split_sentiment.csv"); split["Character"] = split["Character"].str.lower().str.strip()
split["group"] = split["Participant"].map(lambda s:int(next(c for c in str(s) if c.isdigit())))
gt = pd.read_csv("results/step1/01__ground_truth_group_level.csv"); gt["Character"] = gt["Character"].str.lower().str.strip()
logo = LeaveOneGroupOut()
def cv(valcol, target):
    g = split.dropna(subset=[valcol]).groupby(["group","Character","Run"])[valcol].mean().reset_index()
    m = gt[["group","Character","Run",target]].merge(g, on=["group","Character","Run"]).dropna()
    if len(m) < 8: return np.nan, np.nan, len(m)
    r2 = cross_val_score(LinearRegression(), m[[valcol]].values, m[target].values, groups=m["group"].values, cv=logo, scoring="r2").mean()
    return round(r2,3), round(spearmanr(m[valcol], m[target])[0],3), len(m)
print("cv-R2 / rho (leave-one-group-out):")
for valcol in ["val_descriptive","val_stance"]:
    for target in ["positive emotion","like"]:
        r2,rho,n = cv(valcol, target)
        print(f"  {valcol:16s} -> {target:16s} : cv-R2={r2}  rho={rho}  (n={n})")
print("\nPREDICTION: val_descriptive -> positive emotion strong; val_stance -> like stronger than whole-text (0.07).")
print("Caveat: stance sentences are ~20% of text and sparse per cell, so val_stance is noisier -- interpret with the n.")


cv-R2 / rho (leave-one-group-out):
  val_descriptive  -> positive emotion : cv-R2=0.252  rho=0.543  (n=120)
  val_descriptive  -> like             : cv-R2=0.052  rho=0.39  (n=120)
  val_stance       -> positive emotion : cv-R2=0.23  rho=0.516  (n=120)
  val_stance       -> like             : cv-R2=-0.007  rho=0.293  (n=120)

PREDICTION: val_descriptive -> positive emotion strong; val_stance -> like stronger than whole-text (0.07).
Caveat: stance sentences are ~20% of text and sparse per cell, so val_stance is noisier -- interpret with the n.


## Reading it

- **val_descriptive -> positive emotion (high), val_stance -> like (higher than the 0.07 whole-text floor):** confirms the dissociation is about *sentence type*, and that verbalized feeling is recoverable. Strong result.
- **val_stance -> like stays ~0:** then liking really isn't verbalized even when people speak in the first person here (consistent with the 'thoughts about the character' prompt), and the whole-text null is not just dilution.
- Either outcome is informative and directly answers 'can transcribed speech capture the viewer's feeling.'
- **Upgrade path:** replace the rule-based classifier with the LLM hook; score all 6 Step-0 models, not just the winner.